In [ ]:
import panel as pn

pn.extension()

The ``AnyWidget`` pane renders [anywidget](https://anywidget.dev) instances natively in Panel. Instead of relying on the ipywidgets comm infrastructure, it extracts the widget's ESM source and traitlet definitions, creates a dynamic ``AnyWidgetComponent`` subclass, and renders it through Panel's native ReactiveESM pipeline.

This gives you full Panel reactivity — `param.watch`, `pn.bind`, and `.rx` — for any anywidget.

#### Parameters:

For details on other options for customizing the component see the [layout](../../how_to/layout/index.md) and [styling](../../how_to/styling/index.md) how-to guides.

* **``object``** (anywidget instance): The anywidget to render.

___

## Basic Usage

Define a simple counter widget using the anywidget API and wrap it with ``pn.pane.AnyWidget``:

In [ ]:
import anywidget
import traitlets

class CounterWidget(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
        let btn = document.createElement("button");
        btn.style.fontSize = "18px";
        btn.style.padding = "8px 20px";
        btn.style.cursor = "pointer";
        btn.innerHTML = `count is ${model.get("value")}`;
        btn.addEventListener("click", () => {
            model.set("value", model.get("value") + 1);
            model.save_changes();
        });
        model.on("change:value", () => {
            btn.innerHTML = `count is ${model.get("value")}`;
        });
        el.appendChild(btn);
    }
    export default { render };
    """
    value = traitlets.Int(0).tag(sync=True)

widget = CounterWidget()
pane = pn.pane.AnyWidget(widget)
pane

## Auto-detection with `pn.panel()`

Panel automatically detects anywidget instances, so you can use `pn.panel()` directly:

In [ ]:
widget2 = CounterWidget(value=10)
pn.panel(widget2)

## Bidirectional Sync

Changes to the anywidget's traitlets are reflected in the Panel component and vice versa. Access the internal component via `pane.component`:

In [ ]:
# Change the traitlet from Python — the browser updates
widget.value = 42
print(f"widget.value = {widget.value}")
print(f"pane.component.value = {pane.component.value}")

## Reactivity with `param.watch` and `pn.bind`

Since the component exposes proper `param` parameters, you can use Panel's reactive APIs:

In [ ]:
class SliderWidget(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
        let input = document.createElement("input");
        input.type = "range";
        input.min = 0;
        input.max = 100;
        input.value = model.get("value");
        input.style.width = "300px";
        let label = document.createElement("span");
        label.innerHTML = ` Value: ${model.get("value")}`;
        input.addEventListener("input", () => {
            model.set("value", parseInt(input.value));
            model.save_changes();
        });
        model.on("change:value", () => {
            input.value = model.get("value");
            label.innerHTML = ` Value: ${model.get("value")}`;
        });
        el.appendChild(input);
        el.appendChild(label);
    }
    export default { render };
    """
    value = traitlets.Int(50).tag(sync=True)

slider_widget = SliderWidget()
slider_pane = pn.pane.AnyWidget(slider_widget)

# Use pn.bind to create a reactive display
def show_value(value):
    return f"### The slider value is: **{value}**"

pn.Column(
    slider_pane,
    pn.bind(show_value, slider_pane.component.param.value),
)

## Widget with CSS Styling

Anywidgets can include CSS via the `_css` attribute:

In [ ]:
class StyledWidget(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
        let card = document.createElement("div");
        card.className = "styled-card";
        card.innerHTML = `<h3>${model.get("title")}</h3><p>${model.get("content")}</p>`;
        model.on("change:title", () => {
            card.querySelector("h3").textContent = model.get("title");
        });
        model.on("change:content", () => {
            card.querySelector("p").textContent = model.get("content");
        });
        el.appendChild(card);
    }
    export default { render };
    """
    _css = """
    .styled-card {
        border: 2px solid #3b82f6;
        border-radius: 8px;
        padding: 16px;
        background: linear-gradient(135deg, #eff6ff, #dbeafe);
        max-width: 300px;
    }
    .styled-card h3 { color: #1e40af; margin: 0 0 8px; }
    .styled-card p  { color: #1e3a5f; margin: 0; }
    """
    title = traitlets.Unicode("Hello Panel").tag(sync=True)
    content = traitlets.Unicode("AnyWidget + Panel = ❤️").tag(sync=True)

pn.pane.AnyWidget(StyledWidget())